In [1]:
import joblib
data = joblib.load("../scripts/graph_data_coarse_model_20bin_27-12-2024.joblib")

In [2]:
from collections import Counter, defaultdict
import pandas as pd
import numpy as np

def get_mzs(data):
    mz_counts = []
    pairs = list(data["separation_data"].keys())
    print(len(pairs))
    categories = list(set([p[0] for p in pairs]))
    result = {}
    intensities = {}
    for category in categories:
        mzs = []
        intensities1 = {}
        intensities2 = {}
        for pair in pairs:
            if pair[0] == category:
                for x in data["separation_data"][pair]:
                    #if x[1] > 0.8:
                    #if x[2] < 0.01 and ((x[3] / (x[4] + 1e-5) > 3) and x[3] < 3000) or x[3] > 3000:
                    if x[1] > 0.7 and x[2] < 0.01 and (  (((x[3] / (x[4] + 1e-5) > 3)) and x[3] > 1000) or x[3] > 5000):
                        mzs.append(x[0])
                        intensities1[x[0]] = x[3]
                        intensities2[x[0]] = x[4]

                #mzs.extend([x[0] for x in data["separation_data"][pair]])
        c = Counter(mzs)
        most_common_mzs = c.most_common(100)
        
        top_10_mzs = [mz for mz, count in most_common_mzs]
        result[category] = top_10_mzs
        intensities[category] = intensities1, intensities2

        mz_counts.extend(top_10_mzs)

    mz_counts = Counter(mz_counts)
    return result, mz_counts, intensities
        
result, mz_counts, intensities = get_mzs(data)

unique_mzs = defaultdict(list)
common_mzs = defaultdict(list)

csv = defaultdict(list)

for category in result:

    #if not "BST_SNC" in category:
    #if not "PLQ" in category:
    #if not "BG" in category:
    #    continue


    th = 30
    for mz in result[category]:
        if mz in result["BG_ARTEFACT_ARTEFACT"]:
            continue
        
        print(mz, mz_counts[mz], np.percentile(list(mz_counts.values()), 90), np.percentile(list(mz_counts.values()), 10), np.percentile(list(mz_counts.values()), 50))
        if mz_counts[mz] < th:
            unique_mzs[category].append(f"{mz:.2f}")
        else:
            print("common", mz, intensities[category][0][mz], intensities[category][1][mz])
            common_mzs[category].append(f"{mz:.2f}")


for category in unique_mzs:

    csv["category"].append(category)
    csv["unique"].append(','.join(unique_mzs[category][:10]))
    csv["common"].append(','.join(common_mzs[category][:10]))
    
    if "CNU" in category:
        print(category)
        print(list(map(float, unique_mzs[category][:10])))
        print(list(map(float, common_mzs[category][:10])))

csv = pd.DataFrame.from_dict(csv)
#print(csv)
csv.to_csv("mzs_20bin_30-12-2024.csv", index=False)


15006
786.55 65 64.70000000000005 1.0 5.0
common 786.55 28453.928 10045.889
838.55 55 64.70000000000005 1.0 5.0
common 838.55 34489.707 22127.297
787.55 63 64.70000000000005 1.0 5.0
common 787.55 15527.91 5723.2964
839.55 54 64.70000000000005 1.0 5.0
common 839.55 18092.188 11543.667
742.55 65 64.70000000000005 1.0 5.0
common 742.55 25025.936 12548.814
743.55 61 64.70000000000005 1.0 5.0
common 743.55 12589.805 6084.8887
528.3 59 64.70000000000005 1.0 5.0
common 528.3 16414.967 13412.448
699.5 51 64.70000000000005 1.0 5.0
common 699.5 12826.667 6896.926
529.3 32 64.70000000000005 1.0 5.0
common 529.3 5553.0244 4353.897
909.55 57 64.70000000000005 1.0 5.0
common 909.55 7614.252 1476.1482
478.3 75 64.70000000000005 1.0 5.0
common 478.3 25529.113 14470.68
479.3 60 64.70000000000005 1.0 5.0
common 479.3 7500.577 4037.44
863.55 67 64.70000000000005 1.0 5.0
common 863.55 7302.13 5548.407
814.55 51 64.70000000000005 1.0 5.0
common 814.55 6134.61 4163.28
812.55 41 64.70000000000005 1.0 5.0
com

In [3]:
from msi_visual.normalization import total_ion_count, spatial_total_ion_count
from PIL import Image
import numpy as np
from msi_visual.extraction import get_extraction_mz_list
import cv2
from skimage.color import rgb2hed, hed2rgb
import os

def get_ion_image(img, mzs, category, intensity=None):
    indices = [extraction_mzs.index(float(mz)) for mz in mzs]
    mask = img.max(axis=-1)
    img = img / np.max(img, axis=(0, 1))
    ion = img[:, :, indices].mean(axis=-1)
    if intensity is None:
        ion = ion / ion.max()
    else:
        ion = ion / intensity
    #ion = np.uint8(ion * 255)


    ion = ion / np.percentile(ion, 100)
    ion[ion > 1] = 1


    # Apply non-linear stretching to emphasize larger values
    # Using power function with exponent < 1 to compress lower values and stretch higher ones
    ion = np.power(ion, 3)  # Adjust exponent as needed (smaller = more stretching)
    ion = np.uint8((ion / ion.max()) * 255)  # Rescale back to 0-255 range
    #ion = np.uint8(ion* 255)  # Rescale back to 0-255 range

    #display(Image.fromarray(ion))


    # Convert grayscale to RGB IHC-like coloring
    # Create RGB image with brown for high values and light pink for low values
    rgb = np.zeros((ion.shape[0], ion.shape[1], 3), dtype=np.uint8)
    
    # Brown color (RGB: 139, 69, 19) for high values
    # Light pink (RGB: 255, 228, 225) for low values
    rgb[:,:,0] = np.uint8(255 - ion * 0.45)  # R channel 
    rgb[:,:,1] = np.uint8(228 - ion * 0.62)  # G channel
    rgb[:,:,2] = np.uint8(225 - ion * 0.81)  # B channel


    # Create a colormap from white to brown
    white = np.array([255, 255, 255])
    brown = np.array([139, 69, 19]) 
    
    # Create normalized intensity values between 0 and 1
    norm_ion = ion.astype(float) / 255
    
    # For each pixel, interpolate between white and brown based on intensity
    for i in range(3):  # RGB channels
        rgb[:,:,i] = np.uint8(white[i] + (brown[i] - white[i]) * norm_ion)

    rgb[mask == 0] = 0

    
    # This creates a brownish-purple tone typical of IHC staining
    ion = rgb
    # Add text overlay with category name
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.8
    font_color = (255, 50, 60)  # White text
    font_thickness = 2
    text_size = cv2.getTextSize(category, font, font_scale, font_thickness)[0] * 2
    
    # Position text at top center
    ##text_x = (ion.shape[1]) // 2
    text_x = 10
    text_y = text_size[1] + 10  # Add some padding from top

    # Rotate image 90 degrees counterclockwise
    ion = cv2.rotate(ion, cv2.ROTATE_90_COUNTERCLOCKWISE)

    ion_no_text = ion.copy()
    # Add text to image
    category = category.replace("_ARTEFACT", "")
    cv2.putText(ion, category, (text_x, text_y), font, font_scale, font_color, font_thickness)
    return ion, ion_no_text

#5bins
#extraction_mzs = get_extraction_mz_list(r"E:\MSImaging-data\_msi_visual\Extractions\NRL4485-s2\5_bins")
#paths = [(r"E:\MSImaging-data\_msi_visual\Extractions\NRL4485-s2\5_bins\0.npy", 1),
#         (r"E:\MSImaging-data\_msi_visual\Extractions\NRL4485-s2\5_bins\2.npy", 0),
#         (r"E:\MSImaging-data\_msi_visual\Extractions\NRL4485-s2\5_bins\1.npy", 2),
#         (r"E:\MSImaging-data\_msi_visual\Extractions\NRL4485-s2\5_bins\3.npy", 3)]

#10bins
#extraction_mzs = get_extraction_mz_list(r"E:\MSImaging-data\_msi_visual\Extractions\NRL4485-s2\10_bins")
#paths = [(r"E:\MSImaging-data\_msi_visual\Extractions\NRL4485-s2\10_bins\0.npy", 1),
#            (r"E:\MSImaging-data\_msi_visual\Extractions\NRL4485-s2\10_bins\2.npy", 0),
#            (r"E:\MSImaging-data\_msi_visual\Extractions\NRL4485-s2\10_bins\1.npy", 2),
#            (r"E:\MSImaging-data\_msi_visual\Extractions\NRL4485-s2\10_bins\3.npy", 3)]


#20bins
extraction_mzs = get_extraction_mz_list(r"E:\MSImaging-data\_msi_visual\Extractions\NRL4485-s2\20_bins")
paths = [(r"E:\MSImaging-data\_msi_visual\Extractions\NRL4485-s2\20_bins\0.npy", 1),
            (r"E:\MSImaging-data\_msi_visual\Extractions\NRL4485-s2\20_bins\2.npy", 0),
            (r"E:\MSImaging-data\_msi_visual\Extractions\NRL4485-s2\20_bins\1.npy", 2),
            (r"E:\MSImaging-data\_msi_visual\Extractions\NRL4485-s2\20_bins\3.npy", 3)]#


for path, path_index in paths:

    img = np.load(path)
    img = total_ion_count(img)

    images = []
    aggregated_folder = None
    aggregated_folder = "aggregated_per_category_20bin"
    os.makedirs(aggregated_folder, exist_ok=True)
    
    for category in list(common_mzs.keys()):
        
        mzs = common_mzs[category][:10]
        print("common", list(map(float, mzs)))
        intensities_norm = []
        print(intensities[category])
        for mz in mzs:
            mz = float(mz)
            intensities_norm.append((intensities[category][0][mz]))
        intensities_norm = np.mean(intensities_norm)
        
        ion, ion_no_text = get_ion_image(img, mzs, category, intensity=None)
        ion_common = ion.copy()
        print(category, mzs)
        Image.fromarray(ion).save(f"{aggregated_folder}/{category.replace('_ARTEFACT', '')}_{path_index}_common.png")
        Image.fromarray(ion_no_text).save(f"{aggregated_folder}/{category.replace('_ARTEFACT', '')}_{path_index}_common_no_text.png")

        mzs = unique_mzs[category][:10]
        print("unique", list(map(float, mzs)))


        intensities_norm = []
        for mz in mzs:
            mz = float(mz)
            intensities_norm.append((intensities[category][0][mz]))
        intensities_norm = np.mean(intensities_norm)


        ion, ion_no_text = get_ion_image(img, mzs, category, intensity=None)
        Image.fromarray(ion).save(f"{aggregated_folder}/{category.replace('_ARTEFACT', '')}_{path_index}_unique.png")
        Image.fromarray(ion_no_text).save(f"{aggregated_folder}/{category.replace('_ARTEFACT', '')}_{path_index}_unique_no_text.png")


        #images.append(ion)
        
        #display(Image.fromarray(np.hstack([ion_common, ion])))

    # # Create a gallery layout of the images
    # num_images = len(images)
    # num_cols = 1
    # num_rows = (num_images + num_cols - 1) // num_cols  # Ceiling division

    # # Create a blank canvas for the gallery
    # cell_height = images[0].shape[0]
    # cell_width = images[0].shape[1]
    # gallery = np.zeros((cell_height * num_rows, cell_width * num_cols, 3), dtype=np.uint8)

    # # Place each image in the gallery
    # for idx, img in enumerate(images):
    #     i = idx // num_cols  # Row index
    #     j = idx % num_cols   # Column index
        
    #     # Calculate position
    #     y_start = i * cell_height
    #     y_end = (i + 1) * cell_height
    #     x_start = j * cell_width 
    #     x_end = (j + 1) * cell_width
        
    #     gallery[y_start:y_end, x_start:x_end] = img

    # Display the gallery
    #gallery = cv2.resize(gallery, (gallery.shape[1] // 1, gallery.shape[0] // 1))
    #display(Image.fromarray(gallery))
    #Image.fromarray(gallery).save(f"aggregated_per_category/{path_index}.png")






common [786.55, 838.55, 787.55, 839.55, 742.55, 743.55, 528.3, 699.5, 529.3, 909.55]
({658.85: 4853.187, 786.55: 28453.928, 838.55: 34489.707, 1045.5: 5041.138, 787.55: 15527.91, 839.55: 18092.188, 788.55: 76469.16, 766.1: 2128.4553, 789.55: 39305.527, 909.55: 7614.252, 558.05: 2742.569, 906.65: 11656.658, 904.6: 9612.073, 878.6: 11524.35, 863.55: 7302.13, 1046.5: 2912.0894, 907.65: 6485.8457, 696.8: 1314.5447, 812.55: 7986.073, 890.65: 14791.391, 888.6: 20271.748, 742.55: 25025.936, 1023.5: 2235.6584, 888.65: 15868.464, 886.55: 57359.098, 860.65: 17876.643, 814.55: 6134.61, 887.55: 22007.578, 885.55: 104849.39, 879.6: 6107.5283, 891.65: 7638.1055, 862.65: 26865.3, 882.55: 6042.3335, 1047.5: 1668.4227, 522.0: 1073.5366, 861.65: 10223.716, 889.65: 14299.626, 888.55: 7042.6343, 863.65: 14553.366, 506.0: 1162.6342, 844.6: 5134.764, 743.55: 12589.805, 890.6: 5806.016, 810.55: 13339.366, 881.5: 5109.7236, 864.65: 12132.366, 848.65: 12859.675, 346.05: 14951.309, 889.6: 6947.862, 865.65: 5501

C:\Users\PahnkeLab\AppData\Local\Temp\ipykernel_84144\2939713346.py:12: RuntimeWarning: invalid value encountered in divide
  img = img / np.max(img, axis=(0, 1))


BST_MB_VTA ['786.55', '838.55', '787.55', '839.55', '742.55', '743.55', '528.30', '699.50', '529.30', '909.55']
unique [794.6, 822.6]
common [1132.85, 1133.85, 766.55, 767.55, 768.55, 844.55, 746.5, 769.55, 788.5, 762.5]
({658.85: 6484.4395, 766.1: 2461.6548, 696.8: 2025.2393, 834.55: 210690.55, 835.55: 127336.28, 883.55: 28366.502, 836.55: 49628.984, 558.05: 2838.4863, 884.55: 16532.152, 882.55: 7388.4077, 886.55: 69673.54, 885.55: 131187.78, 1045.5: 3719.7764, 887.55: 25700.549, 810.55: 15793.463, 837.55: 16843.992, 506.0: 1611.6, 578.9: 1076.5725, 522.0: 1186.5411, 1132.85: 10437.78, 811.55: 10221.561, 1046.5: 2416.5881, 838.55: 25418.506, 426.0: 3353.6667, 766.55: 137866.17, 888.55: 7490.094, 767.55: 68417.02, 1133.85: 7093.392, 786.55: 15461.157, 965.5: 6695.4434, 812.55: 7143.3843, 882.5: 5334.1294, 746.5: 28103.074, 839.55: 12831.922, 1047.5: 1426.0745, 565.05: 1104.2313, 346.05: 14259.64, 787.55: 8390.564, 857.55: 10763.996, 906.65: 8755.726, 442.0: 1274.545, 907.65: 5174.745, 

c:\Users\PahnkeLab\miniconda3\envs\maldi\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\PahnkeLab\miniconda3\envs\maldi\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\PahnkeLab\AppData\Local\Temp\ipykernel_84144\2939713346.py:13: RuntimeWarning: Mean of empty slice.
  ion = img[:, :, indices].mean(axis=-1)
c:\Users\PahnkeLab\miniconda3\envs\maldi\Lib\site-packages\numpy\core\_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
C:\Users\PahnkeLab\AppData\Local\Temp\ipykernel_84144\2939713346.py:28: RuntimeWarning: invalid value encountered in cast
  ion = np.uint8((ion / ion.max()) * 255)  # Rescale back to 0-255 range


common [861.65, 1112.9, 860.65, 848.65, 863.65, 850.65, 862.65, 864.65, 865.65, 478.3]
({788.55: 120944.02, 789.55: 59932.17, 834.6: 35565.633, 844.65: 26166.414, 846.65: 21596.121, 848.65: 22754.684, 860.65: 33156.074, 861.65: 20056.072, 862.65: 52514.0, 863.65: 29617.488, 864.65: 24037.049, 865.65: 11237.634, 878.6: 17777.121, 889.65: 25758.902, 890.65: 27797.879, 904.6: 15198.78, 906.65: 19543.537, 478.3: 44654.438, 836.6: 14890.415, 891.65: 14110.78, 888.6: 40197.88, 817.6: 8935.78, 849.65: 12634.415, 907.65: 11247.22, 845.65: 15572.585, 847.65: 12477.098, 850.65: 10882.317, 879.6: 10058.073, 816.6: 9966.22, 892.65: 8228.342, 908.65: 5838.073, 844.6: 8074.1465, 888.65: 21891.488, 1112.9: 12102.658, 506.3: 20105.707, 905.6: 7070.317, 479.3: 12451.195, 866.65: 4190.5366, 862.6: 9917.22, 863.55: 7311.5366, 814.55: 8170.3413, 835.65: 10843.536, 701.5: 41236.88, 880.6: 5915.732, 842.6: 7467.122, 1113.9: 8948.415, 1045.5: 4150.488, 845.6: 4269.7803, 835.6: 13107.78, 837.65: 6229.122, 507

In [16]:
gallery = cv2.resize(gallery, (gallery.shape[1] // 2, gallery.shape[0] // 2))
display(Image.fromarray(gallery))



NameError: name 'gallery' is not defined

In [123]:
for pair in data["separation_data"]:
    if "BST_SNC" in pair[0] or "BST_SNC" in pair[1]:
        mzs = [a[0] for a in data["separation_data"][pair]]
        if 697.8 in mzs:
            index = mzs.index(697.8)
            print(pair, data["separation_data"][pair][index])



('BST_SNC_ARTEFACT', 'BG_ARTEFACT_ARTEFACT') (697.8, 0.8343851132686084, 0.004153644170936878)
('BST_SNC_ARTEFACT', 'BST_HY_ARTEFACT') (697.8, 0.9320987654320988, 0.0004817499856699182)
('BST_SNC_ARTEFACT', 'BST_HY_LHA') (697.8, 0.8873239436619719, 0.0006289453187800903)
('BST_SNC_ARTEFACT', 'BST_HY_PH') (697.8, 0.8525641025641025, 0.004177479500536606)
('BST_SNC_ARTEFACT', 'BST_HY_PSTN') (697.8, 0.8888888888888888, 0.003783924642491073)
('BST_SNC_ARTEFACT', 'BST_HY_RCH') (697.8, 0.8333333333333334, 0.010351003203090004)
('BST_SNC_ARTEFACT', 'BST_HY_STN') (697.8, 0.8771929824561404, 0.002957933715796373)
('BST_SNC_ARTEFACT', 'BST_HY_SUB1') (697.8, 0.8333333333333334, 0.010069302314067281)
('BST_SNC_ARTEFACT', 'BST_HY_SUM') (697.8, 0.8611111111111112, 0.007448202179861139)
('BST_SNC_ARTEFACT', 'BST_HY_ZI') (697.8, 0.8971631205673759, 0.0005315919543415871)
('BST_SNC_ARTEFACT', 'BST_MB_APN SUB1') (697.8, 0.8915857605177994, 0.0007074057642417861)
('BST_SNC_ARTEFACT', 'BST_MB_APN SUB2') (

In [1]:
# write a list of the unique mzs - > txt

mzs = [832.60] #paste list here

unique_mzs = sorted(list(set(mzs)))
with open('unique_mzs.txt', 'w') as f:
    f.write(','.join(str(mz) for mz in unique_mzs))
